### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-20b", model_provider = "groq")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots “talk” because they’re exceptionally good at **mimicking sounds**—a skill that evolved for social and ecological reasons. Here’s how it works:\n\n| What they do | Why it matters | How it’s possible |\n|--------------|----------------|-------------------|\n| **Copy human and animal vocalizations** | Parrots live in highly social groups where vocal signals coordinate breeding, territory, and group cohesion. Mimicking lets them blend into the social fabric. | They have a highly flexible **syrinx** (the bird equivalent of a larynx) that can produce a wide range of pitches and timbres. |\n| **Use mimicry to attract mates** | A male that can produce varied, complex calls shows the female he’s fit and intelligent. | The syrinx can modulate airflow and vocal tract shape, giving fine control over sound. |\n| **Respond to environmental cues** | In the wild, parrots mimic alarm calls, predator sounds, or other species to warn or communicate. | Their brains are wired to 

In [ ]:
from langchain.tools import tool
@tool
def get_weather(loaction:str)->str:
    """Get the weather at a loaction"""
    return f"It's Sunny in {loaction}"


model_with_tools = model.bind_tools([get_weather])   

## Note: model with tool call only generated the ai reasoning and not the actual output we want.
## to get the output we need a execution loop/

In [12]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We have a function get_weather. We should call the function with loaction: "Boston". Let\'s call it.', 'tool_calls': [{'id': 'fc_d5a5b948-d3b4-4fd3-88c5-9a8d82288801', 'function': {'arguments': '{"loaction":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 129, 'total_tokens': 189, 'completion_time': 0.067487651, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.007190018, 'prompt_tokens_details': None, 'queue_time': 0.364904688, 'total_time': 0.074677669}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ed70b23331', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1f778-2ba5-74ca-ae3d-5a3f770e1a90-0' tool_calls=[{'name': 'get_weather', 'args': {'loaction': 'Boston'}, 'id': 'fc_d5a5b948-d3b4-4fd

### Tool Execution Loops

In [13]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston!
